# BERT-tiny Quantization with Comprexx

This notebook shows how to compress a small transformer model:

1. Profile the model
2. Apply low-rank decomposition to shrink Linear layers
3. Apply weight-only INT4 quantization
4. Benchmark before/after

We use a minimal 2-layer transformer so this runs in seconds on CPU.

Install: `pip install comprexx`

In [ ]:
import torch.nn as nn

import comprexx as cx

## 1. Define a small transformer

2-layer encoder, d_model=128, 4 heads, feedforward dim=512. Small enough for a notebook, large enough to show compression working.

In [ ]:
class TinyBERT(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, nhead=4, num_layers=2, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=512, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.encoder(x)
        # Mean pooling over sequence dim
        x = x.mean(dim=1)
        return self.classifier(x)

model = TinyBERT()
model.eval()
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

## 2. Profile the model

We pass token IDs as input, but the profiler needs a float tensor. We'll profile using the embedding output shape and note that the embedding table is counted in params.

In [ ]:
# For profiling, we use a float input that skips the embedding.
# The full model takes integer token IDs, so we profile the encoder+classifier separately.
class EncoderClassifier(nn.Module):
    """Wraps encoder + classifier with float input for profiling."""
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, x):
        x = self.encoder(x)
        return self.classifier(x.mean(dim=1))

profiling_model = EncoderClassifier(model.encoder, model.classifier)
profile = cx.analyze(profiling_model, input_shape=(1, 32, 128))  # (batch, seq_len, d_model)
print(profile.summary())

## 3. Low-rank decomposition

The feedforward layers inside the transformer encoder are 128x512 and 512x128. SVD can factorize these into pairs of smaller layers. We keep 50% of the singular values (by energy).

In [ ]:
from comprexx.stages.base import StageContext

stage_lr = cx.stages.LowRankDecomposition(mode="energy", energy_threshold=0.9)
ctx = StageContext(input_shape=(1, 32, 128), device="cpu")

model_lr, report_lr = stage_lr.apply(profiling_model, ctx)
print(report_lr.summary())

## 4. Weight-only INT4 quantization

After SVD, we quantize remaining Linear weights to INT4 (group size 64, symmetric). Activations stay in float32.

In [ ]:
stage_wq = cx.stages.WeightOnlyQuant(bits=4, group_size=64, symmetric=True)

model_quant, report_wq = stage_wq.apply(model_lr, ctx)
print(report_wq.summary())

## 5. Or use a Pipeline for the same thing

In [ ]:
pipeline = cx.Pipeline([
    cx.stages.LowRankDecomposition(mode="energy", energy_threshold=0.9),
    cx.stages.WeightOnlyQuant(bits=4, group_size=64),
])

result = pipeline.run(profiling_model, input_shape=(1, 32, 128))
print(result.report.summary())

## 6. Benchmark latency

In [ ]:
cmp = cx.compare_benchmarks(
    profiling_model, result.model,
    input_shape=(1, 32, 128),
    warmup=10,
    iters=50,
)
print(cmp.summary())

## 7. Dynamic PTQ as an alternative

If weight-only quantization isn't giving you enough speedup, dynamic PTQ quantizes both weights and activations to INT8 at runtime. It's a simpler path that works well for inference on CPU.

In [ ]:
pipeline_ptq = cx.Pipeline([
    cx.stages.LowRankDecomposition(mode="energy", energy_threshold=0.9),
    cx.stages.PTQDynamic(),
])

result_ptq = pipeline_ptq.run(profiling_model, input_shape=(1, 32, 128))
print(result_ptq.report.summary())

cmp_ptq = cx.compare_benchmarks(
    profiling_model, result_ptq.model,
    input_shape=(1, 32, 128),
    warmup=10,
    iters=50,
)
print("\n" + cmp_ptq.summary())